# درس ۲ — آماده‌سازی داده و تعیین ورودی/هدف مدل
## پروژه تخمین هزینه حمل Atramart

در درس قبل دیتاست را شناختیم. در این درس تصمیم می‌گیریم **مدل چه اطلاعاتی ببیند** و **چه چیزی را پیش‌بینی کند**.

هدف پروژه: پیش‌بینی هزینه واقعی حمل (`real_cost`).

> دیتاست فعلی مصنوعی و آموزشی است؛ فعلاً روی یادگیری روش کار می‌کنیم.

## ۱. کتابخانه‌ها و خواندن داده

`pandas` جدول داده را مدیریت می‌کند. `Path` کمک می‌کند مسیر فایل را به‌شکل مرتب بسازیم.

اگر این نوت‌بوک در پوشه `notebooks/` و CSV در `data/raw/` باشد، مسیر زیر درست است.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/atramart_random_shipping_dataset.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["shipping_date"])

print("ابعاد دیتاست:", df.shape)
df.head()

ابعاد دیتاست: (3000, 13)


,order_id,weight,items_count,base_city,destination_city,destination_region,product_category,shipping_date,package_count,carrier_name,service_type,shipping_cost,real_cost
0,ATR-000001,1381,2,Tehran,Yazd,NaN,Computer Case,2025-01-24,1,Post,Special,187016,227872
1,ATR-000002,1644,4,Tehran,Yazd,NaN,Monitor,2025-05-14,1,Courier,Normal,183119,197091
2,ATR-000003,1345,8,Tehran,Tehran,North,Laptop,2025-09-11,2,Courier,Normal,161537,162420
3,ATR-000004,4924,1,Tehran,Tehran,South,Networking,2026-06-03,1,Courier,Express,173595,170093
4,ATR-000005,296,6,Tehran,Tabriz,NaN,Monitor,2026-05-04,2,Tipax,Normal,175357,217311


## ۲. فرق ویژگی (Feature) و هدف (Target)

- **ویژگی یا ورودی (`X`)**: اطلاعاتی که مدل هنگام پیش‌بینی در اختیار دارد.
- **هدف (`y`)**: مقداری که مدل باید یاد بگیرد پیش‌بینی کند.

در پروژه ما هدف، هزینه واقعی حمل است؛ یعنی ستون `real_cost`.

ستون `order_id` شناسه است و الگوی واقعی هزینه را توضیح نمی‌دهد، پس ورودی مدل نمی‌شود.

همچنین فعلاً `shipping_cost` را حذف می‌کنیم: این مبلغ قبلاً از مشتری دریافت شده و ممکن است از سیاست قیمت‌گذاری قبلی تأثیر گرفته باشد. اگر آن را وارد کنیم، مدل می‌تواند به قیمت قبلی تکیه کند، نه فقط به عوامل هزینه حمل.

In [2]:
TARGET = "real_cost"

FEATURES = [
    "weight",
    "items_count",
    "base_city",
    "destination_city",
    "destination_region",
    "product_category",
    "shipping_date",
    "package_count",
    "carrier_name",
    "service_type",
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("تعداد ویژگی‌ها:", X.shape[1])
print("شکل X:", X.shape)
print("شکل y:", y.shape)
X.head()

تعداد ویژگی‌ها: 10
شکل X: (3000, 10)
شکل y: (3000,)


,weight,items_count,base_city,destination_city,destination_region,product_category,shipping_date,package_count,carrier_name,service_type
0,1381,2,Tehran,Yazd,NaN,Computer Case,2025-01-24,1,Post,Special
1,1644,4,Tehran,Yazd,NaN,Monitor,2025-05-14,1,Courier,Normal
2,1345,8,Tehran,Tehran,North,Laptop,2025-09-11,2,Courier,Normal
3,4924,1,Tehran,Tehran,South,Networking,2026-06-03,1,Courier,Express
4,296,6,Tehran,Tabriz,NaN,Monitor,2026-05-04,2,Tipax,Normal


## ۳. بررسی داده‌های ورودی و هدف

قبل از آموزش باید ببینیم آیا مقدار خالی داریم یا نه.

- `X.isna().sum()` تعداد مقدارهای خالی هر ویژگی را می‌شمارد.
- `y.isna().sum()` مقدارهای خالی ستون هدف را می‌شمارد.

در این مرحله فقط بررسی می‌کنیم؛ هنوز مقدار خالی را پر یا حذف نمی‌کنیم. تصمیم درباره نحوه برخورد با داده ناقص را بعداً با دلیل می‌گیریم.

In [3]:
print("مقادیر خالی در ویژگی‌ها:")
print(X.isna().sum())

print("\nمقادیر خالی در هدف:", y.isna().sum())

مقادیر خالی در ویژگی‌ها:
weight                   0
items_count              0
base_city                0
destination_city         0
destination_region    1957
product_category         0
shipping_date            0
package_count            0
carrier_name             0
service_type             0
dtype: int64

مقادیر خالی در هدف: 0


## ۴. بررسی نوع داده‌ها

مدل‌های یادگیری ماشین معمولاً برای محاسبه به ورودی‌های عددی نیاز دارند. اما دیتاست ما ستون‌های متنی هم دارد، مثل `destination_city` و `carrier_name`.

فعلاً آن‌ها را به عدد تبدیل نمی‌کنیم؛ فقط نوعشان را می‌بینیم. در درس بعد یاد می‌گیریم چگونه متن‌های دسته‌ای را به شکل قابل استفاده برای مدل تبدیل کنیم.

In [4]:
X.dtypes

weight                         int64
items_count                    int64
base_city                        str
destination_city                 str
destination_region               str
product_category                 str
shipping_date         datetime64[us]
package_count                  int64
carrier_name                     str
service_type                     str
dtype: object

## ۵. جدا کردن داده آموزش و آزمون

می‌خواهیم مدل را روی بخشی از داده‌ها آموزش دهیم و سپس روی سفارش‌هایی ارزیابی کنیم که هنگام آموزش ندیده است.

- **Training set**: داده‌ای که مدل از آن یاد می‌گیرد.
- **Test set**: داده‌ای که برای سنجش عملکرد نهایی کنار می‌گذاریم.

در این تمرین ۸۰٪ داده برای آموزش و ۲۰٪ برای آزمون می‌گذاریم. `random_state=42` باعث می‌شود تقسیم‌بندی با اجرای دوباره کد قابل تکرار باشد.

این مرحله هنوز مدل را آموزش نمی‌دهد؛ فقط داده‌ها را جدا می‌کند.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2400, 10)
X_test : (600, 10)
y_train: (2400,)
y_test : (600,)


## ۶. تمرین پایان درس

۱. چرا `real_cost` در `X` قرار نگرفته است؟
۲. چرا `order_id` و `shipping_cost` را در فهرست `FEATURES` نگذاشتیم؟
۳. اگر `test_size=0.20` باشد، چه سهمی از داده برای آزمون کنار گذاشته می‌شود؟
۴. خروجی `X_train.shape` و `X_test.shape` را یادداشت کن.

**توقف اینجا:** این درس فقط انتخاب ورودی و هدف و تقسیم داده را انجام می‌دهد. هنوز داده‌های متنی را تبدیل نکرده‌ایم و هنوز مدل آموزش نداده‌ایم. بعد از اجرای سلول‌ها و بررسی خروجی‌ها، درس بعدی را شروع می‌کنیم.